# 캠퍼스 픽 — 교내 게시판 수집 데모

충남대 **본부(plus.cnu.ac.kr)**, **컴퓨터인공지능학부(computer.cnu.ac.kr)**, **경영학부(biz.cnu.ac.kr)** 게시판을 수집해 JSON으로 저장한다.

- 수집 대상: `crawler/sites.json` (환경변수 `CAMPUS_PICK_SITES_FILE`로 다른 파일 지정 가능)
- 방식 2가지, 같은 JSON 스키마로 저장 → 나중에 결과 비교
  - `html`: 공개 게시판 HTML 수집 (지금 사용)
  - `api`: 학교 공식 게시판 API (`CNU_KEY` 발급 후 사용)
- 출력: `data/{html|api}/{site}.json`, `data/{html|api}/_report.json`
- 요청 간격 1초, 이미 받은 글은 본문을 다시 요청하지 않는다. 이메일·전화번호는 마스킹한다.

준비: `pip install -r requirements.txt` 후 이 노트북을 `core/` 폴더에서 연다.

In [ ]:
import json, logging
import pandas as pd
from crawler.collect import collect, compare, load_posts
from crawler.common import load_sites
from crawler import cnu_api

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 60)

## 0. 수집 대상

In [ ]:
sites = load_sites()
pd.DataFrame([{"site": s["site"], "사이트": s["name"], "parser": s["parser"], "robots 준수": s.get("respect_robots", True),
               "board": b["board"], "게시판": b["name"], "API 번호": b.get("api_board_no"), "본문 수집": b.get("fetch_body", True)}
              for s in sites for b in s["boards"]])

## 1. HTML 수집

`LIMIT` = 게시판별 최신 글 수(상단 고정 공지는 별도로 모두 포함). 처음 실행은 글마다 본문을 받아 1~2분 걸리고, 다시 실행하면 새 글만 받는다.

In [ ]:
LIMIT = 5
report = collect("html", limit=LIMIT)
pd.DataFrame([{"site": s, "board": b, **r} for s, v in report["sites"].items() for b, r in v["boards"].items()])

In [ ]:
posts = load_posts("html")
df = pd.DataFrame(posts)
print(f"총 {len(df)}건, HTTP 요청 {report.get('requests')}회")
df.groupby(["site_name", "board_name"]).agg(글수=("id", "size"), 고정공지=("pinned", "sum"),
    본문없음=("content", lambda c: int(sum(not x for x in c))), 첨부=("attachments", lambda a: int(sum(map(len, a)))))

In [ ]:
df.sort_values("posted_at", ascending=False)[["site", "board", "posted_at", "pinned", "title", "url"]].head(15)

### 저장된 글 1건 (JSON 스키마)

In [ ]:
sample = next(p for p in posts if p.get("content"))
print(json.dumps({**sample, "content": sample["content"][:200] + "…"}, ensure_ascii=False, indent=1))

## 2. 공식 API 수집 (`CNU_KEY` 발급 후)

키는 코드·노트북에 적지 않고 환경변수로 넣는다(예: PowerShell `$env:CNU_KEY="..."` 후 Jupyter 실행).
본부 게시판의 API 번호는 아직 모른다 → 키 발급 후 `cnu_api.list_boards()`로 찾아 `sites.json`의 `api_board_no`에 적는다.

In [ ]:
if cnu_api.get_key():
    api_report = collect("api")
    display(pd.DataFrame([{"site": s, "board": b, **r} for s, v in api_report["sites"].items() for b, r in v["boards"].items()]))
    # 본부 게시판 번호 찾기: pd.DataFrame(cnu_api.list_boards()).query("site_nm.str.contains('충남대')")
else:
    print("CNU_KEY 없음 → API 수집 건너뜀")

## 3. HTML vs API 비교

게시판별로 `html_only`(API에 없는 글), `api_only_in_window`(HTML 수집 기간 안인데 HTML에 없는 글), `title_diff`(제목 불일치)를 본다.

In [ ]:
rows = compare()
if load_posts("api"):
    display(pd.DataFrame(rows))
else:
    print("API 결과가 없어 비교 생략 (2번을 먼저 실행)")